# Lab 3: Logical planning
Source: [logic_lab_ex.pdf](logic_lab_ex.pdf). Tasks follow the sheet's order, including the optional extension.

LLM assistance: Codex prepared the code and concise written answers; saved outputs record executed checks.


## Task 0: Understand the planning problem
Initial state: `{At(Robot,A), At(Package,A)}`. Goal: `{At(Package,C)}`.
Connections: `A-B` and `B-C`, in both directions. Absent facts are false; there is one robot and one package.

| Action | Positive preconditions | Negative preconditions | Add | Delete |
|---|---|---|---|---|
| Move(X,Y), for connected X,Y | At(Robot,X) | None | At(Robot,Y) | At(Robot,X) |
| PickUp(Package,X) | At(Robot,X), At(Package,X) | Holding(Package) | Holding(Package) | At(Package,X) |
| Drop(Package,X) | At(Robot,X), Holding(Package) | None | At(Package,X) | Holding(Package) |

Initially `Move(A,B)` and `PickUp(Package,A)` are applicable.
Pickup at A satisfies both positive preconditions. Drop at C fails because the robot is at A and holds nothing.
The pickup's negative precondition explicitly prevents picking up an already-held package.

**Think about it:** availability alone is insufficient; all positive facts must hold and all negative preconditions must be absent.


## Task 1: Construct a plan by hand

| State | Action just applied | Facts |
|---|---|---|
| S0 | None | At(Robot,A), At(Package,A) |
| S1 | PickUp(Package,A) | At(Robot,A), Holding(Package) |
| S2 | Move(A,B) | At(Robot,B), Holding(Package) |
| S3 | Move(B,C) | At(Robot,C), Holding(Package) |
| S4 | Drop(Package,C) | At(Robot,C), At(Package,C) |

S4 contains the goal fact. Moving without pickup would leave the package at A.


## Task 2: Implement the planner
**Implementation prompt used for this LLM-assisted submission:**

> Implement a readable Python planner for one robot and one package in connected locations A-B-C. Represent states as immutable sets of propositions and actions with names, positive/negative preconditions, and add/delete effects. An action is applicable only when its positive preconditions are present and negative preconditions absent. Apply delete effects before add effects. Use BFS over applicable actions to achieve At(Package,C) from At(Robot,A), At(Package,A). Detect No plan found, prevent repeated states, and print the plan and every reached state. Use only Move along the given connections, PickUp, and Drop; reject invalid action application.


In [1]:
from collections import deque
from dataclasses import dataclass


@dataclass(frozen=True)
class Action:
    name: str
    positive: frozenset
    negative: frozenset
    add: frozenset
    delete: frozenset

    def applicable(self, state):
        return self.positive <= state and self.negative.isdisjoint(state)

    def apply(self, state):
        if not self.applicable(state):
            raise ValueError(f"Preconditions fail: {self.name}")
        return (state - self.delete) | self.add


def at(entity, location):
    return f"At({entity},{location})"


holding = "Holding(Package)"
initial = frozenset({at("Robot", "A"), at("Package", "A")})
goal = frozenset({at("Package", "C")})
connections = [("A", "B"), ("B", "A"), ("B", "C"), ("C", "B")]
actions = []
for source, target in connections:
    actions.append(Action(
        f"Move({source},{target})", frozenset({at("Robot", source)}), frozenset(),
        frozenset({at("Robot", target)}), frozenset({at("Robot", source)})))
for location in "ABC":
    actions.append(Action(
        f"PickUp(Package,{location})",
        frozenset({at("Robot", location), at("Package", location)}), frozenset({holding}),
        frozenset({holding}), frozenset({at("Package", location)})))
    actions.append(Action(
        f"Drop(Package,{location})", frozenset({at("Robot", location), holding}), frozenset(),
        frozenset({at("Package", location)}), frozenset({holding})))


def plan_bfs(initial, goal, actions):
    frontier = deque([initial])
    parent = {initial: None}
    while frontier:
        state = frontier.popleft()
        if goal <= state:
            plan = []
            while parent[state] is not None:
                previous, action = parent[state]
                plan.append(action)
                state = previous
            return plan[::-1]
        for action in actions:
            if action.applicable(state):
                successor = action.apply(state)
                if successor not in parent:
                    parent[successor] = (state, action)
                    frontier.append(successor)
    return None


def show_plan(initial, plan):
    print("S0:", ", ".join(sorted(initial)))
    if plan is None:
        print("No plan found")
        return
    state = initial
    for step, action in enumerate(plan, 1):
        state = action.apply(state)
        print(f"{step}. {action.name} -> {', '.join(sorted(state))}")


plan = plan_bfs(initial, goal, actions)
show_plan(initial, plan)


S0: At(Package,A), At(Robot,A)
1. PickUp(Package,A) -> At(Robot,A), Holding(Package)
2. Move(A,B) -> At(Robot,B), Holding(Package)
3. Move(B,C) -> At(Robot,C), Holding(Package)
4. Drop(Package,C) -> At(Package,C), At(Robot,C)


**Think about it:** `applicable` checks preconditions; `apply` updates facts; `goal <= state` terminates planning; the queue explores plans breadth first.

## Task 3: Test the generated planner
Verification recomputes transitions directly from the action fields and checks the single-robot/single-package invariants.
Test C adds a redundant robot-only move and explicitly checks that moving the robot to C leaves the package goal unsatisfied.


In [2]:
def verify_plan(initial, goal, plan):
    if plan is None:
        return False
    state = initial
    for action in plan:
        assert action.positive <= state
        assert action.negative.isdisjoint(state)
        state = (state - action.delete) | action.add
        assert sum(at("Robot", place) in state for place in "ABC") == 1
        assert sum(at("Package", place) in state for place in "ABC") + (holding in state) == 1
    return goal <= state


robot_only = Action("MoveRobotOnly(A,B)", frozenset({at("Robot", "A")}), frozenset(),
                    frozenset({at("Robot", "B")}), frozenset({at("Robot", "A")}))
tests = {
    "A: Solvable": actions,
    "B: No pickup": [a for a in actions if not a.name.startswith("PickUp")],
    "C: Irrelevant robot move": actions + [robot_only],
}
for name, available in tests.items():
    result = plan_bfs(initial, goal, available)
    valid = verify_plan(initial, goal, result)
    print(name)
    print("Initial:", sorted(initial), "Goal:", sorted(goal))
    print("Plan found:", result is not None)
    print("Plan:", [a.name for a in result] if result is not None else "No plan found")
    print("Valid:", valid if result is not None else "No returned plan")
    assert (result is None) if name.startswith("B") else valid
    print()

lookup = {a.name: a for a in actions}
robot_at_c = lookup["Move(B,C)"].apply(lookup["Move(A,B)"].apply(initial))
assert at("Robot", "C") in robot_at_c and at("Package", "A") in robot_at_c
assert not goal <= robot_at_c
assert plan_bfs(initial, goal, [a for a in actions if a.name.startswith("Move")]) is None
assert len(plan) == 4
assert not lookup["PickUp(Package,A)"].applicable(initial | {holding})
try:
    lookup["Drop(Package,C)"].apply(initial)
except ValueError:
    print("Invalid drop rejected; robot-at-C alone does not satisfy the goal.")
else:
    raise AssertionError("Invalid drop was accepted")
print("All planner tests passed")


A: Solvable
Initial: ['At(Package,A)', 'At(Robot,A)'] Goal: ['At(Package,C)']
Plan found: True
Plan: ['PickUp(Package,A)', 'Move(A,B)', 'Move(B,C)', 'Drop(Package,C)']
Valid: True

B: No pickup
Initial: ['At(Package,A)', 'At(Robot,A)'] Goal: ['At(Package,C)']
Plan found: False
Plan: No plan found
Valid: No returned plan

C: Irrelevant robot move
Initial: ['At(Package,A)', 'At(Robot,A)'] Goal: ['At(Package,C)']
Plan found: True
Plan: ['PickUp(Package,A)', 'Move(A,B)', 'Move(B,C)', 'Drop(Package,C)']
Valid: True

Invalid drop rejected; robot-at-C alone does not satisfy the goal.
All planner tests passed


## Task 4: Logic and search
Logic checks whether an action is applicable and determines its successor facts. BFS searches among those legal successors for a goal state.

```text
Current state -> Check preconditions -> Apply applicable action's effects
              -> Generate successor -> Search alternatives -> Goal satisfied?
```

**Think about it:** logic defines possible steps; search chooses which sequences to explore.

## Task 5 (optional): Verify the explanation
**Verification prompt:**

> For every action in the four-step plan, identify its preconditions and show that they hold in the preceding state. Compare the explanation with independently executed transitions.

Pickup requires both objects at A; the two moves require the robot at their source; drop requires the robot at C and a held package. The trace below checks these claims.


In [3]:
state = initial
for action in plan:
    positive_ok = action.positive <= state
    negative_ok = action.negative.isdisjoint(state)
    print(action.name, "requires", sorted(action.positive), "absent", sorted(action.negative),
          "satisfied:", positive_ok and negative_ok)
    assert positive_ok and negative_ok
    state = (state - action.delete) | action.add
assert goal <= state
print("Goal verified:", sorted(state))


PickUp(Package,A) requires ['At(Package,A)', 'At(Robot,A)'] absent ['Holding(Package)'] satisfied: True
Move(A,B) requires ['At(Robot,A)'] absent [] satisfied: True
Move(B,C) requires ['At(Robot,B)'] absent [] satisfied: True
Drop(Package,C) requires ['At(Robot,C)', 'Holding(Package)'] absent [] satisfied: True
Goal verified: ['At(Package,C)', 'At(Robot,C)']


Executed transitions provide stronger evidence than an explanation because they test the specified action semantics. The action definitions themselves still need review.

## Reflection questions
1. Explicit preconditions and effects remove ambiguity and define executable correctness checks.
2. Without a holding precondition, the robot could drop a package it never picked up.
3. A reasonable-looking sequence may contain an inapplicable action or fail the actual package goal.
4. Codex generated the representation, BFS implementation, and tests.
5. Execution checked applicability, effects, invariants, failure handling, and the final goal.
6. Logical reasoning appears in fact-set precondition checks and state updates.
7. Planning searches a graph of symbolic states; actions generate its edges.


## Task 6 (optional): Prolog as a plan verifier
Create `planner.pl` and query the connection rules using SWI-Prolog.


In [4]:
from pathlib import Path
import shutil
import subprocess

prolog_file = Path("planner.pl")
prolog_file.write_text("""connected(a,b).
connected(b,a).
connected(b,c).
connected(c,b).

can_move(X,Y) :- connected(X,Y).
""", encoding="utf-8")
swipl = shutil.which("swipl")
if swipl is None:
    raise RuntimeError("Install SWI-Prolog and put swipl on PATH for the optional tasks.")


def prolog_query(query):
    command = [swipl, "-q", "-s", str(prolog_file), "-g",
               f"({query} -> write(true) ; write(false)), halt"]
    result = subprocess.run(command, capture_output=True, text=True, check=True, timeout=10)
    answer = result.stdout.strip()
    print(f"?- {query}. {answer}.")
    return answer == "true"


assert prolog_query("can_move(a,b)")
assert not prolog_query("can_move(a,c)")


?- can_move(a,b). true.
?- can_move(a,c). false.


(a) `connected(a,b)` is a fact matching the rule body. (b) There is no direct `connected(a,c)` fact and the rule does not compute reachability.
(c) `can_move(X,Y) :- connected(X,Y)` expresses `Connected(X,Y) -> CanMove(X,Y)` for this rule fragment.

## Task 7 (optional): Check a proposed movement plan


In [5]:
with prolog_file.open("a", encoding="utf-8") as stream:
    stream.write("\nvalid_move(X,Y) :- connected(X,Y).\n")
assert prolog_query("valid_move(a,b)")
assert prolog_query("valid_move(b,c)")
assert not prolog_query("valid_move(a,c)")
assert prolog_query("valid_move(a,b), valid_move(b,c)")
print("Proposed Move(a,c) rejected by the warehouse knowledge base.")


?- valid_move(a,b). true.
?- valid_move(b,c). true.


?- valid_move(a,c). false.
?- valid_move(a,b), valid_move(b,c). true.
Proposed Move(a,c) rejected by the warehouse knowledge base.


**Think about it:** these queries independently check movement connectivity; they do not verify package pickup, dropping, or the entire planning state.

## Task 8 (optional): Connect Prolog to logical reasoning


In [6]:
with prolog_file.open("a", encoding="utf-8") as stream:
    stream.write("\nwet_road.\nslippery :- wet_road.\nreduce_speed :- slippery.\n")
assert prolog_query("reduce_speed")


?- reduce_speed. true.


`WetRoad` (fact), `WetRoad -> Slippery` (rule), and `Slippery -> ReduceSpeed` (rule) imply `ReduceSpeed` (conclusion).

**Think about it:** facts and rules let Prolog infer query answers. Its operational mechanisms include unification and backtracking; Prolog is not identical to classical logic.

## Optional-extension reflection
1. A fact asserts something directly; a rule derives a head when its body succeeds.
2. A query asks whether the program can establish a proposition through its facts and rules.
3. Prolog provides a separate check that proposed movements match the warehouse connections.
4. A separate verifier can expose unsupported actions even when the generator's explanation sounds plausible.
